In [28]:
from langchain_community.vectorstores import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_core.documents import Document

In [29]:
embedding = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-2"
)

In [30]:
# Step 1: Your source documents
documents = [
    Document(page_content="LangChain helps developers build LLM applications easily."),
    Document(page_content="Chroma is a vector database optimized for LLM-based search."),
    Document(page_content="Embeddings convert text into high-dimensional vectors."),
    Document(page_content="OpenAI provides powerful embedding models."),
    Document(page_content="Code gemini provides data science course.")
]

In [31]:
from langchain_chroma import Chroma

vector_store = Chroma.from_documents(

documents=documents,

embedding=embedding,

persist_directory="genai07"

)


In [32]:
retriever = vector_store.as_retriever(search_kwargs={"k": 2})

In [33]:
query = "What is Chroma used for?"
results = retriever.invoke(query)

In [34]:
results

[Document(id='6714a6ca-f6eb-46f9-9d08-828c35cf928f', metadata={}, page_content='Chroma is a vector database optimized for LLM-based search.'),
 Document(id='588894aa-8ff4-4c81-8bee-2310137dad71', metadata={}, page_content='LangChain helps developers build LLM applications easily.')]

In [35]:
for i, doc in enumerate(results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)


--- Result 1 ---
Chroma is a vector database optimized for LLM-based search.

--- Result 2 ---
LangChain helps developers build LLM applications easily.


In [36]:
query = "Which course is discussed?"
results = retriever.invoke(query)

In [37]:
results

[Document(id='c94a3dda-30ab-48bf-a336-a33844971379', metadata={}, page_content='Code gemini provides data science course.'),
 Document(id='ece3a8b1-6a80-44fc-8b36-d77f9ebf4e94', metadata={}, page_content='Embeddings convert text into high-dimensional vectors.')]

In [ ]:
# MMR

In [38]:
# Sample documents
docs = [
    Document(page_content="LangChain makes it easy to work with LLMs."),
    Document(page_content="LangChain is used to build LLM based applications."),
    Document(page_content="Chroma is used to store and search document embeddings."),
    Document(page_content="Embeddings are vector representations of text."),
    Document(page_content="MMR helps you get diverse results when doing similarity search."),
    Document(page_content="LangChain supports Chroma, FAISS, Pinecone, and more."),
]

In [40]:
from langchain_community.vectorstores import FAISS

# Initialize OpenAI embeddings
embedding_model = embedding

# Step 2: Create the FAISS vector store from documents
vectorstore = FAISS.from_documents(
    documents=docs,
    embedding=embedding_model
)

In [43]:
results

[Document(id='80dd6b05-452c-462f-a048-49eb7cf03ff4', metadata={}, page_content='LangChain is used to build LLM based applications.'),
 Document(id='4466cbf3-1b30-4fb9-8509-72776d037b41', metadata={}, page_content='MMR helps you get diverse results when doing similarity search.'),
 Document(id='b183d139-fdeb-41da-807c-c0bef6732827', metadata={}, page_content='Chroma is used to store and search document embeddings.')]

In [42]:
query = "What is langchain?"
results = retriever.invoke(query)

In [44]:
for i, doc in enumerate(results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)


--- Result 1 ---
LangChain is used to build LLM based applications.

--- Result 2 ---
MMR helps you get diverse results when doing similarity search.

--- Result 3 ---
Chroma is used to store and search document embeddings.


In [45]:
from langchain_community.vectorstores import FAISS
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.documents import Document
from langchain_classic.retrievers import MultiQueryRetriever
import os

In [46]:
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
llm1 = ChatGoogleGenerativeAI(
        model="gemini-2.5-flash",
        google_api_key=GEMINI_API_KEY,
    )

In [47]:
llm1.invoke('hi')

AIMessage(content='Hi there! How can I help you today?', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019fdc66-b96b-7b92-88ad-9430853b5776-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 2, 'output_tokens': 31, 'total_tokens': 33, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 21}})

In [48]:
# Relevant health & wellness documents
all_docs = [
    Document(page_content="Regular walking boosts heart health and can reduce symptoms of depression.", metadata={"source": "H1"}),
    Document(page_content="Consuming leafy greens and fruits helps detox the body and improve longevity.", metadata={"source": "H2"}),
    Document(page_content="Deep sleep is crucial for cellular repair and emotional regulation.", metadata={"source": "H3"}),
    Document(page_content="Mindfulness and controlled breathing lower cortisol and improve mental clarity.", metadata={"source": "H4"}),
    Document(page_content="Drinking sufficient water throughout the day helps maintain metabolism and energy.", metadata={"source": "H5"}),
    Document(page_content="The solar energy system in modern homes helps balance electricity demand.", metadata={"source": "I1"}),
    Document(page_content="Python balances readability with power, making it a popular system design language.", metadata={"source": "I2"}),
    Document(page_content="Photosynthesis enables plants to produce energy by converting sunlight.", metadata={"source": "I3"}),
    Document(page_content="The 2022 FIFA World Cup was held in Qatar and drew global energy and excitement.", metadata={"source": "I4"}),
    Document(page_content="Black holes bend spacetime and store immense gravitational energy.", metadata={"source": "I5"}),
]

In [49]:
# Create FAISS vector store
vectorstore = FAISS.from_documents(documents=all_docs, embedding=embedding)

In [50]:
# Create retrievers
similarity_retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 5})

In [51]:
multiquery_retriever = MultiQueryRetriever.from_llm(
    retriever=vectorstore.as_retriever(search_kwargs={"k": 5}),
    llm=llm1
)

In [52]:
# Query
query = "How to improve energy levels and maintain balance?"

In [53]:
# Retrieve results
similarity_results = similarity_retriever.invoke(query)
multiquery_results= multiquery_retriever.invoke(query)

In [54]:
for i, doc in enumerate(similarity_results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)

print("*"*150)

for i, doc in enumerate(multiquery_results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)


--- Result 1 ---
Drinking sufficient water throughout the day helps maintain metabolism and energy.

--- Result 2 ---
Mindfulness and controlled breathing lower cortisol and improve mental clarity.

--- Result 3 ---
Consuming leafy greens and fruits helps detox the body and improve longevity.

--- Result 4 ---
Deep sleep is crucial for cellular repair and emotional regulation.

--- Result 5 ---
Regular walking boosts heart health and can reduce symptoms of depression.
******************************************************************************************************************************************************

--- Result 1 ---
Drinking sufficient water throughout the day helps maintain metabolism and energy.

--- Result 2 ---
Consuming leafy greens and fruits helps detox the body and improve longevity.

--- Result 3 ---
Deep sleep is crucial for cellular repair and emotional regulation.

--- Result 4 ---
Mindfulness and controlled breathing lower cortisol and improve mental cla